<a href="https://colab.research.google.com/github/Dreamingdreamers/multimodal-emotion-recognition/blob/main/notebooks/03_face_fer2013_mobilenet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/Dreamingdreamers/multimodal-emotion-recognition.git
%cd YOUR_REPOSITORY
!git checkout -b colleague-work

Cloning into 'multimodal-emotion-recognition'...
remote: Enumerating objects: 46, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 46 (delta 24), reused 20 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (46/46), 268.14 KiB | 2.00 MiB/s, done.
Resolving deltas: 100% (24/24), done.
[Errno 2] No such file or directory: 'YOUR_REPOSITORY'
/content
fatal: not a git repository (or any of the parent directories): .git


In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torchvision.models import convnext_base, ConvNeXt_Base_Weights

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
        roc_auc_score, classification_report, confusion_matrix, roc_curve, auc
        )
from google.colab import drive

In [2]:
# ==========================================
# STEP 1: MOUNT DRIVE & DEFINE PATHS
# ==========================================
drive.mount('/content/drive')

DATASET_DIR = '/content/drive/MyDrive/Multimodal_Emotion_Project/data/raw/fer2013'
TRAIN_DIR = os.path.join(DATASET_DIR, 'train')
TEST_DIR = os.path.join(DATASET_DIR, 'test')

# Local session directories for fast saving of artifacts during training
PROCESSED_PATH = '/content/data/processed'
MODEL_PATH = '/content/models'

os.makedirs(PROCESSED_PATH, exist_ok=True)
os.makedirs(MODEL_PATH, exist_ok=True)

# Verify directory paths
if not os.path.exists(TRAIN_DIR) or not os.path.exists(TEST_DIR):
    raise FileNotFoundError(
       f"Could not locate 'train' or 'test' folders inside {DATASET_DIR}.\n"
       "Please check your Google Drive path structure."
     )

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using compute device: {device}")

Mounted at /content/drive
Using compute device: cpu


In [3]:
# ==========================================
# STEP 2: DATASET & AUGMENTATION PIPELINE
# ==========================================
# ImageNet Transforms (Upscaling 48x48 images to 224x224 for ConvNeXt)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
# Load Datasets using PyTorch ImageFolder
train_dataset = ImageFolder(root=TRAIN_DIR, transform=train_transform)
test_dataset = ImageFolder(root=TEST_DIR, transform=eval_transform)

target_names = train_dataset.classes
num_classes = len(target_names)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print(f"Loaded {len(train_dataset)} training images and {len(test_dataset)} test images across {num_classes} classes:")
print(f"Class Mapping: {train_dataset.class_to_idx}")

Loaded 28718 training images and 7178 test images across 7 classes:
Class Mapping: {'angry': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'neutral': 4, 'sad': 5, 'surprise': 6}


In [4]:
# ==========================================
# STEP 3: CONVNEXT MODEL & LOSS FUNCTION
# ==========================================
model = convnext_base(weights=ConvNeXt_Base_Weights.DEFAULT)

# Replace Output Head for 7 Facial Emotion Classes
in_features = model.classifier[2].in_features
model.classifier[2] = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(in_features, num_classes)
)

model = model.to(device)

# Label Smoothing CrossEntropy Loss to handle noisy facial expressions
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
epochs = 12
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

Downloading: "https://download.pytorch.org/models/convnext_base-6075fbad.pth" to /root/.cache/torch/hub/checkpoints/convnext_base-6075fbad.pth


100%|██████████| 338M/338M [00:02<00:00, 154MB/s]


In [ ]:
# ==========================================
# STEP 4: FINE-TUNING LOOP WITH VALIDATION
# ==========================================
best_val_acc = 0.0

print("\nStarting ConvNeXt fine-tuning on FER2013...")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in train_loader:
       images, labels = images.to(device), labels.to(device)

       optimizer.zero_grad()
       outputs = model(images)
       loss = criterion(outputs, labels)
       loss.backward()
       optimizer.step()

       running_loss += loss.item() * images.size(0)
       _, preds = torch.max(outputs, 1)
       correct += (preds == labels).sum().item()
       total += labels.size(0)

    scheduler.step()

    epoch_loss = running_loss / total
    epoch_acc = (correct / total) * 100

   # Validation / Test Evaluation per Epoch
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
      for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        val_correct += (preds == labels).sum().item()
        val_total += labels.size(0)

    val_acc = (val_correct / val_total) * 100
    print(f"Epoch [{epoch+1:02d}/{epochs:02d}] - Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.2f}% | Test Acc: {val_acc:.2f}%")
    if val_acc > best_val_acc:
      best_val_acc = val_acc
      torch.save(model.state_dict(), f"{MODEL_PATH}/best_convnext_fer2013.pth")

print(f"\nTraining Complete! Peak Test Accuracy: {best_val_acc:.2f}%")





Starting ConvNeXt fine-tuning on FER2013...


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from torchvision.models import convnext_base, ConvNeXt_Base_Weights

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    roc_auc_score, classification_report, confusion_matrix, roc_curve, auc
)
from google.colab import drive

# ==========================================
# STEP 1: EXTRACT LOCAL ZIP FILE
# ==========================================
DRIVE_PATH = '/content/drive/MyDrive/Multimodal_Emotion_Project'
FER2013_CSV_PATH = f'{DRIVE_PATH}/data/raw/fer2013/fer2013.csv'
PROCESSED_PATH = f'{DRIVE_PATH}/data/processed'
MODEL_PATH = f'{DRIVE_PATH}/models/vision_convnext'




os.makedirs(PROCESSED_PATH, exist_ok=True)
os.makedirs(MODEL_PATH, exist_ok=True)



device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using compute device: {device}")

# STEP 2: DATASET & AUGMENTATION PIPELINE
# ==========================================
EMOTION_MAP = {0: 'Angry', 1: 'Disgust', 2: 'Fear', 3: 'Happy', 4: 'Sad', 5: 'Surprise', 6: 'Neutral'}
target_names = list(EMOTION_MAP.values())

class FER2013Dataset(Dataset):
    def _init_(self, df, transform=None):
       self.df = df
       self.transform = transform

    def _len_(self):
       return len(self.df)

    def _getitem_(self, idx):
       row = self.df.iloc[idx]
       pixels = np.array(row['pixels'].split(), dtype='uint8').reshape(48, 48)
       # Convert Grayscale to 3-Channel RGB for Transfer Learning Backbones
       image = Image.fromarray(pixels).convert('RGB')
       label = int(row['emotion'])
       if self.transform:
          image = self.transform(image)

       return image, label
# ImageNet Transforms & Upscaling to 224x224
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Auto-detect whether ZIP contained a CSV file or Folder-based Images
csv_files = []
for root, dirs, files in os.walk(LOCAL_EXTRACT_DIR):
    for file in files:
        if file.lower().endswith('.csv'):
            csv_files.append(os.path.join(root, file))

if csv_files:
    print(f"Detected CSV Format Dataset at: {csv_files[0]}")

    class FER2013CSVDataset(Dataset):
        def __init__(self, df, transform=None):
            self.df = df
            self.transform = transform

        def __len__(self):
            return len(self.df)

        def __getitem__(self, idx):
            row = self.df.iloc[idx]
            pixels = np.array(row['pixels'].split(), dtype='uint8').reshape(48, 48)
            image = Image.fromarray(pixels).convert('RGB')
            label = int(row['emotion'])

            if self.transform:
                image = self.transform(image)

            return image, label

    df = pd.read_csv(csv_files[0])
    train_df = df[df['Usage'] == 'Training'].reset_index(drop=True)
    val_df = df[df['Usage'] == 'PublicTest'].reset_index(drop=True)
    test_df = df[df['Usage'] == 'PrivateTest'].reset_index(drop=True)

    train_loader = DataLoader(FER2013CSVDataset(train_df, train_transform), batch_size=32, shuffle=True, num_workers=2)
    val_loader = DataLoader(FER2013CSVDataset(val_df, eval_transform), batch_size=64, shuffle=False, num_workers=2)
    test_loader = DataLoader(FER2013CSVDataset(test_df, eval_transform), batch_size=64, shuffle=False, num_workers=2)

    print(f"Dataset Split -> Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

else:
    print("Detected ImageFolder Format Dataset. Loading image directories...")
    from torchvision.datasets import ImageFolder

    train_dir, test_dir = None, None
    for root, dirs, files in os.walk(LOCAL_EXTRACT_DIR):
        for d in dirs:
            if d.lower() in ['train', 'training']:
                train_dir = os.path.join(root, d)
            elif d.lower() in ['test', 'publictest', 'privatetest', 'val']:
                test_dir = os.path.join(root, d)

    train_ds = ImageFolder(train_dir, transform=train_transform)
    test_ds = ImageFolder(test_dir, transform=eval_transform)

    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
    val_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2)

    print(f"Dataset Split -> Train: {len(train_ds)} | Test: {len(test_ds)}")

# ==========================================
# STEP 3: CONVNEXT MODEL & LOSS FUNCTION
# ==========================================
model = convnext_base(weights=ConvNeXt_Base_Weights.DEFAULT)

# Replace Classifier Head for 7 Facial Emotion Classes
in_features = model.classifier[2].in_features
model.classifier[2] = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(in_features, 7)
)

model = model.to(device)

# Label Smoothing CrossEntropy Loss
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
epochs = 12
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

# ==========================================
# STEP 4: FINE-TUNING LOOP WITH VALIDATION
# ==========================================
best_val_acc = 0.0

print("\nStarting ConvNeXt fine-tuning on FER2013...")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    scheduler.step()

    epoch_loss = running_loss / total
    epoch_acc = (correct / total) * 100

    # Validation Step
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_acc = (val_correct / val_total) * 100
    print(f"Epoch [{epoch+1:02d}/{epochs:02d}] - Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.2f}% | Val Acc: {val_acc:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), f"{MODEL_PATH}/best_convnext_fer2013.pth")

print(f"\nTraining Complete! Peak Validation Accuracy: {best_val_acc:.2f}%")

# ==========================================
# STEP 5: EVALUATION & VISUALIZATIONS
# ==========================================
model.load_state_dict(torch.load(f"{MODEL_PATH}/best_convnext_fer2013.pth"))
model.eval()

all_probs = []
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)

        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

probs = np.array(all_probs)
preds = np.array(all_preds)
y_true = np.array(all_labels)

# Calculate Evaluation Metrics
acc = accuracy_score(y_true, preds)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, preds, average='macro', zero_division=0)
auc_roc = roc_auc_score(y_true, probs, multi_class='ovr', average='macro')

print("\n==========================================")
print("     VISION CONVNEXT PERFORMANCE METRICS  ")
print("==========================================")
print(f"1. Accuracy         : {acc * 100:.2f}%")
print(f"2. Precision (Macro): {precision:.4f}")
print(f"3. Recall (Macro)   : {recall:.4f}")
print(f"4. F1-Score (Macro) : {f1:.4f}")
print(f"5. AUC-ROC (Macro)  : {auc_roc:.4f}")
print("==========================================\n")

print("Detailed Classification Report:")
print(classification_report(y_true, preds, target_names=target_names, zero_division=0))

# Plot Confusion Matrix and ROC Curves
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# A. Confusion Matrix
cm = confusion_matrix(y_true, preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', xticklabels=target_names, yticklabels=target_names, ax=axes[0], cbar=False)
axes[0].set_title('Vision Model Confusion Matrix', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Predicted Emotion')
axes[0].set_ylabel('True Emotion')

# B. Multi-Class ROC Curves
colors = plt.cm.get_cmap('tab10', 7)
for i in range(7):
    y_binary = (y_true == i).astype(int)
    fpr, tpr, _ = roc_curve(y_binary, probs[:, i])
    roc_val = auc(fpr, tpr)
    axes[1].plot(fpr, tpr, color=colors(i), lw=2, label=f'{target_names[i]} (AUC = {roc_val:.2f})')

axes[1].plot([0, 1], [0, 1], 'k--', lw=1.5)
axes[1].set_title('Vision Multi-Class ROC Curves (One-vs-Rest)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc="lower right", fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Save Local Artifacts for Fusion
np.save(os.path.join(PROCESSED_PATH, 'vision_test_probs.npy'), probs)
np.save(os.path.join(PROCESSED_PATH, 'vision_test_labels.npy'), y_true)
fig.savefig(os.path.join(PROCESSED_PATH, 'vision_evaluation_plots.png'), dpi=300)

print("\nVision prediction probabilities and evaluation plots saved locally to /content/data/processed/!")

Extracting /content/FER2013.zip to local runtime storage...
Extraction complete!
Using compute device: cpu
Detected ImageFolder Format Dataset. Loading image directories...
Dataset Split -> Train: 28709 | Test: 7178
Downloading: "https://download.pytorch.org/models/convnext_base-6075fbad.pth" to /root/.cache/torch/hub/checkpoints/convnext_base-6075fbad.pth


100%|██████████| 338M/338M [00:03<00:00, 104MB/s]



Starting ConvNeXt fine-tuning on FER2013...


In [ ]:
{
 "cells": [],
 "metadata": {},
 "nbformat": 4,
 "nbformat_minor": 2
}

{'cells': [], 'metadata': {}, 'nbformat': 4, 'nbformat_minor': 2}